In [49]:
%pip install -qq pandas faker factory-boy 


Note: you may need to restart the kernel to use updated packages.


c:\Users\dunn0172\Documents\GitHub\biorepository-data-wrangling\.venv\Scripts\python.exe: No module named pip


In [50]:
import pandas as pd
import factory

In [51]:
class PatientFactory(factory.Factory):
    class Meta:
        model = dict

    patient_id = factory.Sequence(lambda n: f"PT{n:04d}")
    age = factory.Faker('random_int', min=18, max=90)

In [52]:
class StudySiteFactory(factory.Factory):
    class Meta:
        model = dict

    site_id = factory.Sequence(lambda n: f"SITE{n:03d}")
    site_name = factory.Faker('company')
    

In [ ]:
class StudyVisitFactory(factory.Factory):
    class Meta:
        model = dict

    visit_id = factory.Sequence(lambda n: f"VISIT{n:05d}")
    # Values are injected from StudyFactory to ensure concrete IDs, not declarations.
    patient_id = None
    site_id = None
    visit_date = factory.Faker('date_this_year')
    specimen_schema = None

In [90]:
import random

SPECIMEN_TYPES = [
    "EDTA plasma",
    "Serum",
    "Heparin plasma",
    "PAXgene RNA",
    "Buffy coat",
]

def generate_specimen_schema():
    picked_types = random.sample(SPECIMEN_TYPES, k=random.randint(2, 4))
    return [
        {"specimen_type": specimen_type, "count": random.randint(1, 6)}
        for specimen_type in picked_types
    ]

class StudyFactory(factory.Factory):
    class Meta:
        model = dict

    Name = factory.Faker('sentence', nb_words=3)
    StartDate = factory.Faker('date_this_decade')
    EndDate = factory.Faker('date_this_decade')
    Sites = factory.LazyFunction(lambda: [StudySiteFactory() for _ in range(3)])
    Participants = factory.LazyFunction(lambda: [PatientFactory() for _ in range(10)])
    Visits = factory.LazyAttribute(
        lambda obj: [
            StudyVisitFactory(
                patient_id=random.choice([p['patient_id'] for p in obj.Participants]),
                site_id=random.choice([s['site_id'] for s in obj.Sites]),
                specimen_schema=generate_specimen_schema(),
            )
            for _ in range(5)
        ]
    )

In [99]:
# Generate a random dataset
study = StudyFactory()
# Convert to DataFrame for display
study

{'Name': 'Reduce stage.',
 'StartDate': datetime.date(2023, 9, 12),
 'EndDate': datetime.date(2026, 1, 17),
 'Sites': [{'site_id': 'SITE006', 'site_name': 'Gardner, Davis and Moran'},
  {'site_id': 'SITE007', 'site_name': 'Pittman, Phillips and Fox'},
  {'site_id': 'SITE008', 'site_name': 'Harris PLC'}],
 'Participants': [{'patient_id': 'PT0020', 'age': 50},
  {'patient_id': 'PT0021', 'age': 41},
  {'patient_id': 'PT0022', 'age': 70},
  {'patient_id': 'PT0023', 'age': 86},
  {'patient_id': 'PT0024', 'age': 51},
  {'patient_id': 'PT0025', 'age': 22},
  {'patient_id': 'PT0026', 'age': 37},
  {'patient_id': 'PT0027', 'age': 68},
  {'patient_id': 'PT0028', 'age': 62},
  {'patient_id': 'PT0029', 'age': 51}],
 'Visits': [{'visit_id': 'VISIT00010',
   'patient_id': 'PT0027',
   'site_id': 'SITE008',
   'visit_date': datetime.date(2026, 3, 5),
   'specimen_schema': [{'specimen_type': 'Heparin plasma', 'count': 3},
    {'specimen_type': 'EDTA plasma', 'count': 1}]},
  {'visit_id': 'VISIT00011',

In [100]:
import sys
from pathlib import Path

def resolve_duckdb_file() -> str:
    # In JupyterLite/Pyodide, use a local file in the virtual FS.
    if sys.platform == "emscripten":
        return "clinical_study.db"

    # In desktop Python, keep using the repo's content/data location.
    db_path = (Path.cwd() / "../data/clinical_study.db").resolve()
    db_path.parent.mkdir(parents=True, exist_ok=True)
    return str(db_path)

duckdb_file = resolve_duckdb_file()
print(f"DuckDB file: {duckdb_file}")

DuckDB file: C:\Users\dunn0172\Documents\GitHub\biorepository-data-wrangling\content\data\clinical_study.db


In [101]:
# Delete existing database file if it exists
import os
if os.path.exists(duckdb_file):
    os.remove(duckdb_file)

In [114]:
# Write generated study to DuckDB database
import duckdb
# Create a connection to DuckDB
conn = duckdb.connect(duckdb_file)


In [115]:
# Create tables and insert data
conn.execute("""
CREATE TABLE IF NOT EXISTS Study (
    Name TEXT,
    StartDate DATE,
    EndDate DATE
)""")
conn.execute("""CREATE TABLE IF NOT EXISTS StudySite (
    site_id TEXT,
    site_name TEXT
)""")
conn.execute("""CREATE TABLE IF NOT EXISTS Patient (
    patient_id TEXT,
    age INTEGER
)""")
conn.execute("""CREATE TABLE IF NOT EXISTS StudyVisit (
    visit_id TEXT,
    patient_id TEXT,
    site_id TEXT,
    visit_date DATE
)""")
conn.execute("""CREATE TABLE IF NOT EXISTS StudyVisitSpecimen (
    visit_id TEXT,
    specimen_type TEXT,
    specimen_count INTEGER
)""")
conn.execute("""CREATE TABLE IF NOT EXISTS StudySpecimen (
    specimen_id TEXT,
    visit_id TEXT,
    specimen_type TEXT,
    aliquot_number INTEGER
)""")


In [116]:
# Make reruns idempotent by clearing previous synthetic data first.
conn.execute("DELETE FROM StudySpecimen")
conn.execute("DELETE FROM StudyVisitSpecimen")
conn.execute("DELETE FROM StudyVisit")
conn.execute("DELETE FROM Patient")
conn.execute("DELETE FROM StudySite")
conn.execute("DELETE FROM Study")

# Insert study data
conn.execute("INSERT INTO Study (Name, StartDate, EndDate) VALUES (?, ?, ?)", (study['Name'], study['StartDate'], study['EndDate']))
# Insert site data
for site in study['Sites']:
    conn.execute("INSERT INTO StudySite (site_id, site_name) VALUES (?, ?)", (site['site_id'], site['site_name']))
# Insert patient data
for patient in study['Participants']:
    conn.execute("INSERT INTO Patient (patient_id, age) VALUES (?, ?)", (patient['patient_id'], patient['age']))
# Insert visit and specimen data
for visit in study['Visits']:
    conn.execute(
        "INSERT INTO StudyVisit (visit_id, patient_id, site_id, visit_date) VALUES (?, ?, ?, ?)",
        (visit['visit_id'], visit['patient_id'], visit['site_id'], visit['visit_date'])
    )

    for specimen in visit['specimen_schema']:
        conn.execute(
            "INSERT INTO StudyVisitSpecimen (visit_id, specimen_type, specimen_count) VALUES (?, ?, ?)",
            (visit['visit_id'], specimen['specimen_type'], specimen['count'])
        )

        for aliquot_number in range(1, specimen['count'] + 1):
            specimen_prefix = specimen['specimen_type'].upper().replace(' ', '_')
            specimen_id = f"{visit['visit_id']}_{specimen_prefix}_{aliquot_number:02d}"
            conn.execute(
                "INSERT INTO StudySpecimen (specimen_id, visit_id, specimen_type, aliquot_number) VALUES (?, ?, ?, ?)",
                (specimen_id, visit['visit_id'], specimen['specimen_type'], aliquot_number),
            )
# Commit changes and keep the connection open for verification cells.
conn.commit()

In [112]:
# Read data back from DuckDB to verify
tables = conn.execute("SHOW TABLES").fetchdf()
print("Available tables:")
print(tables)

study_df = conn.execute("SELECT * FROM Study").fetchdf()
sites_df = conn.execute("SELECT * FROM StudySite").fetchdf()
patients_df = conn.execute("SELECT * FROM Patient").fetchdf()
visits_df = conn.execute("SELECT * FROM StudyVisit").fetchdf()
specimen_schema_df = conn.execute("SELECT * FROM StudyVisitSpecimen").fetchdf()
specimen_df = conn.execute("SELECT * FROM StudySpecimen ORDER BY specimen_id").fetchdf()

print("Study Data:")
print(study_df)
print("Sites Data:")
print(sites_df)
print("Patients Data:")
print(patients_df)
print("Visits Data:")
print(visits_df)
print("Specimen Schema Data:")
print(specimen_schema_df)
print("Individual Specimen Data (first 20 rows):")
print(specimen_df.head(20))

Available tables:
                 name
0             Patient
1               Study
2           StudySite
3       StudySpecimen
4          StudyVisit
5  StudyVisitSpecimen
Study Data:
            Name  StartDate    EndDate
0  Reduce stage. 2023-09-12 2026-01-17
1  Reduce stage. 2023-09-12 2026-01-17
Sites Data:
   site_id                  site_name
0  SITE006   Gardner, Davis and Moran
1  SITE007  Pittman, Phillips and Fox
2  SITE008                 Harris PLC
3  SITE006   Gardner, Davis and Moran
4  SITE007  Pittman, Phillips and Fox
5  SITE008                 Harris PLC
Patients Data:
   patient_id  age
0      PT0020   50
1      PT0021   41
2      PT0022   70
3      PT0023   86
4      PT0024   51
5      PT0025   22
6      PT0026   37
7      PT0027   68
8      PT0028   62
9      PT0029   51
10     PT0020   50
11     PT0021   41
12     PT0022   70
13     PT0023   86
14     PT0024   51
15     PT0025   22
16     PT0026   37
17     PT0027   68
18     PT0028   62
19     PT0029   51
Visits 

In [117]:
# Compare schema-level counts to generated individual specimen counts
query = """
SELECT
    svs.visit_id,
    svs.specimen_type,
    svs.specimen_count AS schema_count,
    COUNT(ss.specimen_id) AS generated_count
FROM StudyVisitSpecimen svs
LEFT JOIN StudySpecimen ss
    ON svs.visit_id = ss.visit_id
   AND svs.specimen_type = ss.specimen_type
GROUP BY svs.visit_id, svs.specimen_type, svs.specimen_count
ORDER BY svs.visit_id, svs.specimen_type
"""
joined_df = conn.execute(query).fetchdf()
print(joined_df)

# Close connection when all reads are complete.
conn.close()

      visit_id   specimen_type  schema_count  generated_count
0   VISIT00010     EDTA plasma             1                1
1   VISIT00010  Heparin plasma             3                3
2   VISIT00011     EDTA plasma             3                3
3   VISIT00011  Heparin plasma             5                5
4   VISIT00011           Serum             4                4
5   VISIT00012      Buffy coat             6                6
6   VISIT00012     PAXgene RNA             4                4
7   VISIT00012           Serum             5                5
8   VISIT00013      Buffy coat             3                3
9   VISIT00013     EDTA plasma             1                1
10  VISIT00013  Heparin plasma             1                1
11  VISIT00013           Serum             1                1
12  VISIT00014      Buffy coat             4                4
13  VISIT00014     EDTA plasma             6                6
14  VISIT00014  Heparin plasma             4                4
15  VISI

In [ ]:
# Quick QA: pivot one visit to specimen-type columns
conn = duckdb.connect(duckdb_file)

qa_visit_id = conn.execute(
    "SELECT visit_id FROM StudyVisit ORDER BY visit_id LIMIT 1"
).fetchone()[0]

pivot_query = """
SELECT
    ? AS visit_id,
    SUM(CASE WHEN specimen_type = 'EDTA plasma' THEN 1 ELSE 0 END) AS edta_plasma,
    SUM(CASE WHEN specimen_type = 'Serum' THEN 1 ELSE 0 END) AS serum,
    SUM(CASE WHEN specimen_type = 'Heparin plasma' THEN 1 ELSE 0 END) AS heparin_plasma,
    SUM(CASE WHEN specimen_type = 'PAXgene RNA' THEN 1 ELSE 0 END) AS paxgene_rna,
    SUM(CASE WHEN specimen_type = 'Buffy coat' THEN 1 ELSE 0 END) AS buffy_coat
FROM StudySpecimen
WHERE visit_id = ?
"""

qa_pivot_df = conn.execute(pivot_query, (qa_visit_id, qa_visit_id)).fetchdf()
print(qa_pivot_df)

conn.close()